In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")


In [ ]:
!pip install ultralytics -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!unzip -q /content/drive/MyDrive/wheat-split.zip \
  -d /content/
!ls /content/wheat-split/


In [ ]:
import os

# Auto-detect the correct data path by finding 'train' folder
def find_data_root(base='/content'):
    """Walk directory tree to find folder containing train/val/test."""
    for root, dirs, files in os.walk(base):
        if 'train' in dirs and 'val' in dirs:
            return root
    return None

data_path = find_data_root('/content/wheat-split')
if data_path is None:
    data_path = find_data_root('/content')

if data_path is None:
    raise FileNotFoundError("Could not find a folder with train/ and val/ subdirs")

print(f"Found data root: {data_path}")
print(f"  train/ classes: {os.listdir(os.path.join(data_path, 'train'))}")
print(f"  val/   classes: {os.listdir(os.path.join(data_path, 'val'))}")
train_count = sum(len(f) for _, _, f in os.walk(os.path.join(data_path, 'train')))
val_count = sum(len(f) for _, _, f in os.walk(os.path.join(data_path, 'val')))
print(f"  train images: {train_count}  |  val images: {val_count}")

from ultralytics import YOLO
model = YOLO('yolov8n-cls.pt')
# Auto-tune batch/cache to GPU VRAM (avoids OOM on T4)
import torch
_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
_batch = 32 if _vram >= 10 else 16
_cache = "disk" if _vram < 14 else "ram"  # disk avoids OOM on T4
print(f"GPU VRAM: {_vram:.1f} GB  ->  batch={_batch}  cache={_cache}")

results = model.train(
    data=data_path,
    epochs=50,
    imgsz=224,
    batch=_batch,
    device=0,
    project='/content/drive/MyDrive/agridrone-wheat',
    name='wheat_v1',
    seed=42,
    patience=10,
    save=True,
    plots=True,
    augment=True
    cache=_cache,   # disk avoids GPU OOM on T4
    amp=True,        # mixed precision halves VRAM usage
)


In [ ]:
metrics = model.val()
print(f"Top-1 Accuracy: {metrics.top1:.3f}")
print(f"Top-5 Accuracy: {metrics.top5:.3f}")
print("Model saved to Google Drive")
print("Download: agridrone-wheat/wheat_v1/weights/best.pt")


In [ ]:
import glob, os
from google.colab import files

# Auto-find the latest best.pt from any wheat run
candidates = glob.glob('/content/drive/MyDrive/agridrone-wheat/wheat_v*/weights/best.pt')
if not candidates:
    raise FileNotFoundError("No best.pt found in agridrone-wheat/wheat_v*/weights/")

# Pick the most recently modified one
best_path = max(candidates, key=os.path.getmtime)
print(f"Downloading: {best_path}")
print(f"Size: {os.path.getsize(best_path) / 1e6:.1f} MB")
files.download(best_path)
